
# Vietnamese Depression Project — Near-Duplicate / Historical Split Leakage Audit

**Version:** 1.0

This notebook is a thin, reproducible launcher and review interface for the authoritative server script `near_duplicate_leakage_audit.py`.

The audit was added after unsupervised LDA inspection revealed repeated/templated text families that were not caught by exact-duplicate checks. It asks a focused question: **do normalization-equivalent or highly similar records cross the exact historical held-out train/test boundaries?**

The audit uses the two original raw CSVs plus the already-audited preprocessing outputs. It does not alter any existing experiment files.


In [ ]:

from pathlib import Path
import sys, subprocess, json
import pandas as pd

PROJECT_ROOT = Path.cwd()
SCRIPT = PROJECT_ROOT / "near_duplicate_leakage_audit.py"
RESULTS = PROJECT_ROOT / "near_duplicate_audit_v1"

print("Project root:", PROJECT_ROOT)
print("Script exists:", SCRIPT.exists())
print("Raw D1 exists:", (PROJECT_ROOT / "datasets/dataset_1.csv").exists())
print("Raw D2 exists:", (PROJECT_ROOT / "datasets/dataset_2.csv").exists())
print("Audited D1 preprocessing exists:", (PROJECT_ROOT / "reproduction_results_v1/preprocessed/Dataset_depression_vi1.csv").exists())
print("Audited D2 preprocessing exists:", (PROJECT_ROOT / "reproduction_results_v1/preprocessed/Dataset_depression_vi2.csv").exists())



## Run the audit

For long unattended server execution, the README's `nohup python -u ...` command is preferred. This cell is convenient for interactive notebook use.


In [ ]:

RUN_AUDIT = False  # Set True only when you want this notebook to launch the full audit.

if RUN_AUDIT:
    cmd = [
        sys.executable, "-u", str(SCRIPT),
        "--project-root", str(PROJECT_ROOT),
        "--n-jobs", "4",
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)
else:
    print("RUN_AUDIT=False. Use the nohup command in README_NEAR_DUPLICATE_AUDIT.md for the server run.")


## Review compact outputs after completion

In [ ]:

summary_path = RESULTS / "tables/near_duplicate_leakage_summary.csv"
if summary_path.exists():
    summary = pd.read_csv(summary_path)
    display(summary)
else:
    print("Leakage summary not created yet:", summary_path)

sens_path = RESULTS / "tables/existing_prediction_clean_test_sensitivity.csv"
if sens_path.exists():
    sensitivity = pd.read_csv(sens_path)
    display(sensitivity)
else:
    print("Prediction sensitivity file not created yet (or no compatible predictions were found).")



## Interpretation rules

- A high cosine-similarity pair is evidence of textual similarity, not automatic proof of semantic identity.
- Inspect **0.90, 0.95, and 0.98 together**; do not select a threshold after looking at the answer.
- Cross-label near duplicates deserve special manual review because they may indicate annotation/source conflicts.
- If meaningful train→test family overlap is present, the proper follow-up is **group-aware splitting and retraining**.
- The post-hoc "clean-test" metric table is a sensitivity analysis only; it is not a replacement for retraining.
